# Week 4 — Exercise: Chat + parking / restaurant (LangGraph + OpenRouter)

In [ ]:
from __future__ import annotations

import json
import math
import os
from typing import Any, Literal, TypedDict

from pydantic import BaseModel

import httpx
from dotenv import load_dotenv
from IPython.display import Image, Markdown, display
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage, ToolMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.graph import END, START, StateGraph
from langgraph.prebuilt import create_react_agent
import gradio as gr


In [ ]:
load_dotenv(override=True)

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")

def make_llm() -> ChatOpenAI:
    return ChatOpenAI(
        model='openai/gpt-4o-mini',
        temperature=0,
        api_key=OPENROUTER_API_KEY,
        base_url="https://openrouter.ai/api/v1",
    )


In [ ]:
class LocationParseError(BaseModel):
    ok: Literal[False] = False
    reason: str
    latitude: float | None = None
    longitude: float | None = None


class LocationParseSuccess(BaseModel):
    ok: Literal[True] = True
    latitude: float
    longitude: float


def _finalize_coordinates(latitude: float, longitude: float) -> LocationParseSuccess | LocationParseError:
    if not math.isfinite(latitude) or not math.isfinite(longitude):
        return LocationParseError(reason="non_finite", latitude=latitude, longitude=longitude)
    if not (-90.0 <= latitude <= 90.0 and -180.0 <= longitude <= 180.0):
        return LocationParseError(reason="out_of_range", latitude=latitude, longitude=longitude)
    return LocationParseSuccess(latitude=latitude, longitude=longitude)


@tool
def submit_user_coordinates(latitude: float, longitude: float) -> str:
    """Call after you (the agent) extracted WGS84 numbers from the user's message. Validates ranges and returns JSON: ok, latitude, longitude (or ok:false with reason). The app uses ok:true for OpenStreetMap (Overpass) search."""
    return _finalize_coordinates(latitude, longitude).model_dump_json(exclude_none=True)

In [ ]:
OVERPASS_URL = "https://overpass-api.de/api/interpreter"
USER_AGENT = "PR/1.0"

#find the distance between two points on the earth using the haversine formula
def haversine_m(lat1: float, lon1: float, lat2: float, lon2: float) -> float:
    r = 6371000.0
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dl = math.radians(lon2 - lon1)
    a = math.sin(dphi / 2) ** 2 + math.cos(p1) * math.cos(p2) * math.sin(dl / 2) ** 2
    return 2 * r * math.asin(math.sqrt(a))


def _center_from_el(el: dict[str, Any]) -> tuple[float, float] | None:
    if "lat" in el and "lon" in el:
        return float(el["lat"]), float(el["lon"])
    c = el.get("center") or {}
    if "lat" in c and "lon" in c:
        return float(c["lat"]), float(c["lon"])
    return None


def overpass_amenity(lat: float, lng: float, radius_m: int, amenity: str, timeout: int = 25) -> list[dict[str, Any]]:
    q = f"""[out:json][timeout:{timeout}];
(
  node["amenity"="{amenity}"](around:{radius_m},{lat},{lng});
  way["amenity"="{amenity}"](around:{radius_m},{lat},{lng});
);
out center;
"""
    with httpx.Client(timeout=60.0, headers={"User-Agent": USER_AGENT}) as client:
        r = client.post(OVERPASS_URL, data={"data": q})
        r.raise_for_status()
    data = r.json()
    out: list[dict[str, Any]] = []
    for el in data.get("elements", []):
        center = _center_from_el(el)
        if center is None:
            continue
        plat, plng = center
        tags = el.get("tags") or {}
        eid = el.get("id", "?")
        name = tags.get("name") or tags.get("ref") or f"osm#{eid}"
        out.append(
            {
                "name": name,
                "lat": plat,
                "lng": plng,
                "distance_from_ref_m": haversine_m(lat, lng, plat, plng),
            }
        )
    out.sort(key=lambda x: x["distance_from_ref_m"])
    return out


In [ ]:
@tool
def parking_specialist_search_near_user(latitude: float, longitude: float, radius_m: int = 2500) -> str:
    """Parking specialist: list OSM parkings near the user, nearest first."""
    rows = overpass_amenity(latitude, longitude, radius_m, "parking")
    return json.dumps({"parkings": rows, "count": len(rows)})


@tool
def restaurant_specialist_search_near_point(latitude: float, longitude: float, radius_m: int = 800) -> str:
    """Restaurant specialist: list OSM restaurants near a point."""
    rows = overpass_amenity(latitude, longitude, radius_m, "restaurant")
    return json.dumps({"restaurants": rows, "count": len(rows)})


In [ ]:
llm = make_llm()

REPLY_SYSTEM = """You are the **Reply agent** — you talk with the user until you have valid coordinates, then the app sends those coordinates to the parking/restaurant search (OpenStreetMap via Overpass).

- Be warm and concise. On greetings, greet back and offer help finding parking near a restaurant.
- If they ask what you can do, explain and ask if they want that search.
- When they want search, ask for a **Google Maps** link (look for `@lat,lng` or `?q=lat,lng`) or **two decimal numbers** (latitude, longitude in WGS84).
- When their message clearly contains numeric coordinates, **you** read them from the text (order: latitude then longitude for a plain pair; in many map URLs the first number after `@` is latitude). Call **submit_user_coordinates** once with those two floats.
- Never call the tool with guessed coordinates from a city or place name alone — only when numbers appear in the message.
- If the tool returns **ok:false** (e.g. out_of_range), explain and ask for a corrected location.
- If **ok:true**, confirm briefly — the **application** will run the OSM search and append results below your reply.
- Keep replies short unless they ask for detail."""

PARKING_SPECIALIST_PROMPT = """You are the parking specialist. When given latitude and longitude, call "
"parking_specialist_search_near_user once with those numbers. Do not invent coordinates."""

RESTAURANT_SPECIALIST_PROMPT = """You collaborate with the parking specialist. You receive JSON with a list `parkings` sorted by distance from the user. "
"For EACH parking in order, call restaurant_specialist_search_near_point with that parking's lat and lng "
"until a call returns count > 0, or you have tried all parkings. Report which parking index worked."""

reply_agent = create_react_agent(
    llm,
    tools=[submit_user_coordinates],
    prompt=REPLY_SYSTEM,
    name="reply_agent",
)



parking_agent = create_react_agent(
    llm,
    tools=[parking_specialist_search_near_user],
    prompt=PARKING_SPECIALIST_PROMPT,
    name="parking_agent",
)

restaurant_agent = create_react_agent(
    llm,
    tools=[restaurant_specialist_search_near_point],
    prompt=RESTAURANT_SPECIALIST_PROMPT,
    name="restaurant_agent",
)


In [ ]:
def last_tool_content(messages: list, tool_name: str) -> str | None:
    for m in reversed(messages):
        if isinstance(m, ToolMessage) and m.name == tool_name:
            return str(m.content)
    return None


def extract_submitted_coordinates(messages: list) -> dict[str, Any] | None:
    raw = last_tool_content(messages, "submit_user_coordinates")
    if raw is None:
        return None
    return json.loads(raw)


def extract_parking_json(messages: list) -> str | None:
    return last_tool_content(messages, "parking_specialist_search_near_user")


def last_assistant_text(messages: list) -> str:
    for m in reversed(messages):
        if isinstance(m, AIMessage) and (m.content or '').strip():
            return str(m.content)
    return ""


In [ ]:
class SearchState(TypedDict, total=False):
    latitude: float
    longitude: float
    parking_json: str | None
    restaurant_messages: list | None
    final_markdown: str


def node_parking(state: SearchState) -> SearchState:
    lat, lng = state["latitude"], state["longitude"]
    text = (
        f"Find parking options near user latitude {lat} and longitude {lng}. "
        "Call parking_specialist_search_near_user once."
    )
    out = parking_agent.invoke({"messages": [HumanMessage(text)]})
    msgs = out.get('messages', [])
    pj = extract_parking_json(msgs) or "{}"
    return {"parking_json": pj}


def node_restaurant(state: SearchState) -> SearchState:
    lat, lng = state["latitude"], state["longitude"]
    parking_json = state.get("parking_json") or "{}"
    text = (
        f"User is at latitude {lat}, longitude {lng}.\n\n"
        f"Parking specialist JSON:\n{parking_json}\n\n"
        "Try each parking in order: call restaurant_specialist_search_near_point for each parking's lat/lng "
        "until restaurants appear, or all parkings are exhausted."
    )

    out = restaurant_agent.invoke(
        {"messages": [HumanMessage(text)]},
        config={"recursion_limit": 20},
    )
    return {"restaurant_messages": out.get("messages", [])}


def node_summarize(state: SearchState) -> SearchState:
    summary_llm = make_llm()
    rest_msgs = state.get('restaurant_messages') or []
    rest_tail = ''
    for m in rest_msgs[-8:]:
        if isinstance(m, (AIMessage, ToolMessage)):
            rest_tail += type(m).__name__ + ": " + str(getattr(m, "content", "")) + "\n"
    sys = SystemMessage(
        content=(
            "You write concise Markdown for the user. Include: coordinates, "
            "parking summary from JSON, what the restaurant specialist tried, and the best restaurant found "
            "(if any). Mention that data comes from OpenStreetMap and may be incomplete."
        )
    )
    hum = HumanMessage(
        content=(
            f"User at lat={state['latitude']}, lng={state['longitude']}\n\n"
            f"Parking JSON:\n{state.get('parking_json')}\n\n"
            f"Restaurant agent trace (last messages):\n{rest_tail}"
        )
    )
    out = summary_llm.invoke([sys, hum])
    return {"final_markdown": out.content}


def build_search_graph() -> StateGraph:
    g = StateGraph(SearchState)
    g.add_node("parking", node_parking)
    g.add_node("restaurant", node_restaurant)
    g.add_node("summarize", node_summarize)
    g.add_edge(START, "parking")
    g.add_edge("parking", "restaurant")
    g.add_edge("restaurant", "summarize")
    g.add_edge("summarize", END)
    return g.compile()


search_graph = build_search_graph()


In [ ]:
try:
    display(Image(search_graph.get_graph().draw_mermaid_png()))
except Exception as e:
    print("Diagram skipped:", e)


In [ ]:
def messages_from_chat(history: list) -> list:
    msgs = []
    for pair in history or []:
        if not pair or len(pair) < 2:
            continue
        u, a = pair[0], pair[1]
        msgs.append(HumanMessage(str(u)))
        if a is not None and str(a).strip():
            msgs.append(AIMessage(content=str(a)))
    return msgs


def chat_with_agents(message: str, history: list):
    message = (message or '').strip()
    if not message:
        yield history
        return

    hist = list(history or [])
    yield hist + [[message, '*typing…*']]

    prior = messages_from_chat(hist)
    prior.append(HumanMessage(message))

    out = reply_agent.invoke({"messages": prior})
    rmsgs = out.get('messages', [])
    reply_text = last_assistant_text(rmsgs)
    if not reply_text:
        reply_text = "I could not produce a reply."

    parsed = extract_submitted_coordinates(rmsgs)
    if not parsed or not parsed.get('ok'):
        yield hist + [[message, reply_text]]
        return

    lat, lng = float(parsed['latitude']), float(parsed['longitude'])
    last_s: dict | None = None
    try:
        for state in search_graph.stream(
            {"latitude": lat, "longitude": lng},
            stream_mode="values",
        ):
            last_s = state
    except Exception as e:
        yield hist + [[message, reply_text + f"\n\n**Search error:** `{e}`"]]
        return

    search_md = (last_s or {}).get("final_markdown") or "*No search output.*"
    final = reply_text + "\n\n" + search_md
    yield hist + [[message, final]]


with gr.Blocks(title="Find parking near a restaurant") as demo:
    gr.Markdown(
    "Find parking near a restaurant"
    )
    chat = gr.Chatbot(label="Conversation", height=420)
    msg = gr.Textbox(
        placeholder="Type a message…",
        label="Message",
        lines=2,
    )
    send = gr.Button("Send", variant="primary")
    _inputs = [msg, chat]
    _outputs = [chat]
    send.click(chat_with_agents, _inputs, _outputs, show_progress="full").then(
        lambda: "", outputs=msg
    )
    msg.submit(chat_with_agents, _inputs, _outputs, show_progress="full").then(
        lambda: "", outputs=msg
    )

demo.launch(share=False, inline=True)
